In [5]:
"""Build `waiter_level_features.parquet` using client scores from `client_anomaly_scores.csv`.

When `waiter_week_anomaly_scores.csv` exists at the project root (same as `models/waiter_week_models.py`),
adds `share_anomaly_weeks_iso`, `share_anomaly_weeks_ocsvm`, `share_anomaly_weeks_lof` using the same
logic as `models/lala.ipynb` (global score percentiles, flag > 0.95, share over weeks with 3+ distinct weeks),
and `avg_anomaly_score_iso` / `avg_anomaly_score_ocsvm` / `avg_anomaly_score_lof` (mean waiter-week scores per waiter),
plus `max_anomaly_week_score_*` and `median_anomaly_week_score_*` over that waiter's waiter-week rows.

Per distinct client served by the waiter: `max_client_anomaly_score_*`, `median_client_anomaly_score_*`,
`avg_client_anomaly_score_*` from client anomaly scores.

Also adds `num_of_trn_as_top_waiter`, `share_trn_as_top_waiter`, `num_of_clients_as_top_waiter`,
`share_clients_as_top_waiter` (transactions and clients where `waiter_id` equals the client's `top_waiter_id`).
`share_trn_as_top_waiter_5` / `share_trn_as_top_waiter_10` are the same share over the waiter's total distinct
transactions, but the numerator only includes rows where the client has `num_of_trn` > 4 or > 9.

Adds `avg_weighted_anomaly_score_iso_as_top_waiter`, `avg_weighted_anomaly_score_ocsvm_as_top_waiter`,
`avg_weighted_anomaly_score_lof_as_top_waiter`: for clients with `top_waiter_id == waiter_id`,
sum(num_of_trn * anomaly_score) / sum(num_of_trn) per model.

Adds `median_share_top_waiter_as_top_waiter`: median of client `share_top_waiter` over the same clients (those with `top_waiter_id == waiter_id`).
`median_share_top_waiter_as_top_waiter_5` / `_10` use the same relation but only clients with `num_of_trn` > 4 or > 9 (mirroring `share_trn_as_top_waiter_5` / `_10`).

Adds `share_trn_of_places_worked`: for each place, total distinct `trn_id`; for each waiter–place pair, that waiter's distinct trn count.
Pooled share = sum of waiter counts across places where they work / sum of those places' totals (distinct trn).
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd()
ROOT = _here if (_here / "config.py").exists() else _here.parent
if not (ROOT / "config.py").exists():
    raise FileNotFoundError(f"Expected config.py under {ROOT}; run from project root or parquet/")

sys.path.insert(0, str(ROOT))

from config import DATA_PATH, FRAUD_IDS


def _share_anomaly_weeks_from_waiter_week_scores(
    ww: pd.DataFrame,
    percentile_threshold: float = 0.95,
    min_distinct_weeks: int = 3,
) -> pd.DataFrame:
    """
    Match models/lala.ipynb: rank anomaly scores globally across waiter-weeks, flag
    percentile > threshold, then share_anomaly_weeks_* = sum(flags) / n_distinct weeks
    for waiters with more than 2 distinct weeks (else NaN). Also returns mean iso/ocsvm/lof
    scores across that waiter's waiter-week rows.
    """
    d = ww.copy()
    score_cols = ["iso_score", "ocsvm_score", "lof_score"]
    for col in score_cols:
        d[f"{col}_percentile"] = d[col].rank(pct=True, method="average")
    perc_cols = ["iso_score_percentile", "ocsvm_score_percentile", "lof_score_percentile"]
    for pc in perc_cols:
        d[f"anomaly_{pc}"] = d[pc] > percentile_threshold

    agg = d.groupby("waiter_id", as_index=False).agg(
        anomaly_weeks_iso=("anomaly_iso_score_percentile", "sum"),
        anomaly_weeks_ocsvm=("anomaly_ocsvm_score_percentile", "sum"),
        anomaly_weeks_lof=("anomaly_lof_score_percentile", "sum"),
        total_weeks=("week", "nunique"),
        avg_anomaly_score_iso=("iso_score", "mean"),
        avg_anomaly_score_ocsvm=("ocsvm_score", "mean"),
        avg_anomaly_score_lof=("lof_score", "mean"),
        max_anomaly_week_score_iso=("iso_score", "max"),
        max_anomaly_week_score_ocsvm=("ocsvm_score", "max"),
        max_anomaly_week_score_lof=("lof_score", "max"),
        median_anomaly_week_score_iso=("iso_score", "median"),
        median_anomaly_week_score_ocsvm=("ocsvm_score", "median"),
        median_anomaly_week_score_lof=("lof_score", "median"),
    )
    ok = agg["total_weeks"] >= min_distinct_weeks
    tw = agg["total_weeks"].replace(0, np.nan)
    agg["share_anomaly_weeks_iso"] = np.where(
        ok, agg["anomaly_weeks_iso"] / tw, np.nan
    )
    agg["share_anomaly_weeks_ocsvm"] = np.where(
        ok, agg["anomaly_weeks_ocsvm"] / tw, np.nan
    )
    agg["share_anomaly_weeks_lof"] = np.where(
        ok, agg["anomaly_weeks_lof"] / tw, np.nan
    )
    return agg[
        [
            "waiter_id",
            "share_anomaly_weeks_iso",
            "share_anomaly_weeks_ocsvm",
            "share_anomaly_weeks_lof",
            "avg_anomaly_score_iso",
            "avg_anomaly_score_ocsvm",
            "avg_anomaly_score_lof",
            "max_anomaly_week_score_iso",
            "max_anomaly_week_score_ocsvm",
            "max_anomaly_week_score_lof",
            "median_anomaly_week_score_iso",
            "median_anomaly_week_score_ocsvm",
            "median_anomaly_week_score_lof",
        ]
    ]


def _fraud_waiter_ids(df: pd.DataFrame, client_data: pd.DataFrame) -> set:
    cd = client_data
    fraud_waiter_ids = set(
        cd.loc[cd.index.isin(FRAUD_IDS), "top_waiter_id"]
        .dropna()
        .unique()
    )

    # Co-waiters at the same place: 3+ distinct transactions with a fraud person_id.
    fp = df[df["person_id"].isin(FRAUD_IDS)]
    n_by_place_waiter = (
        fp.groupby(["person_id", "place_id", "waiter_id"])["trn_id"]
        .nunique()
        .reset_index(name="n_trn")
    )
    colluders = n_by_place_waiter.loc[
        n_by_place_waiter["n_trn"] >= 3, "waiter_id"
    ].dropna()
    fraud_waiter_ids.update(colluders.unique())

    print("Num of fraud waiter ids: ", len(fraud_waiter_ids))
    return fraud_waiter_ids


def build_waiter_data(df: pd.DataFrame, client_data: pd.DataFrame) -> pd.DataFrame:
    """
    Build waiter-level DataFrame from transaction df and client_data (with anomaly scores).
    client_data must have anomaly_score_iso, anomaly_score_ocsvm, anomaly_score_lof,
    top_waiter_id, share_top_waiter, num_of_trn, days_visits, is_fraud (index: person_id).
    df must have person_id, waiter_id, place_id, date, trn_id.
    Uses distinct trn_id counts, consistent with waiter-level `num_of_trn`.
    """
    if "anomaly_score_iso" not in df.columns and "anomaly_score_iso" in client_data.columns:
        scores = client_data[["anomaly_score_iso", "anomaly_score_ocsvm", "anomaly_score_lof"]]
        df = df.merge(scores, left_on="person_id", right_index=True, how="left")
    elif "anomaly_score_iso" not in df.columns:
        raise ValueError("Anomaly scores must be in df or client_data")

    waiter_data = df.groupby("waiter_id").agg(
        iso_90=("anomaly_score_iso", lambda x: x.quantile(0.9)),
        ocsvm_90=("anomaly_score_ocsvm", lambda x: x.quantile(0.9)),
        lof_90=("anomaly_score_lof", lambda x: x.quantile(0.9)),
        num_of_trn=("trn_id", "nunique"),
        num_of_clients=("person_id", "nunique"),
        working_days=("date", "nunique"),
    ).reset_index()

    client_scores_by_waiter = (
        df[
            [
                "waiter_id",
                "person_id",
                "anomaly_score_iso",
                "anomaly_score_ocsvm",
                "anomaly_score_lof",
            ]
        ]
        .drop_duplicates(["waiter_id", "person_id"])
        .groupby("waiter_id", as_index=False)
        .agg(
            max_client_anomaly_score_iso=("anomaly_score_iso", "max"),
            median_client_anomaly_score_iso=("anomaly_score_iso", "median"),
            avg_client_anomaly_score_iso=("anomaly_score_iso", "mean"),
            max_client_anomaly_score_ocsvm=("anomaly_score_ocsvm", "max"),
            median_client_anomaly_score_ocsvm=("anomaly_score_ocsvm", "median"),
            avg_client_anomaly_score_ocsvm=("anomaly_score_ocsvm", "mean"),
            max_client_anomaly_score_lof=("anomaly_score_lof", "max"),
            median_client_anomaly_score_lof=("anomaly_score_lof", "median"),
            avg_client_anomaly_score_lof=("anomaly_score_lof", "mean"),
        )
    )
    waiter_data = waiter_data.merge(client_scores_by_waiter, on="waiter_id", how="left")

    fraud_waiter_ids = _fraud_waiter_ids(df, client_data)
    waiter_data["is_fraud"] = waiter_data["waiter_id"].isin(fraud_waiter_ids)

    active_person_ids = client_data[
        (client_data["num_of_trn"] > 5) & (client_data["days_visits"] > 5)
    ].index

    active_clients_per_waiter = (
        df[df["person_id"].isin(active_person_ids)]
        .groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_of_active_clients")
    )
    waiter_data = waiter_data.merge(
        active_clients_per_waiter, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_of_active_clients"] = waiter_data["num_of_active_clients"].fillna(0).astype(int)

    person_place_waiter = (
        df.groupby(["person_id", "place_id"])["waiter_id"]
        .nunique()
        .reset_index(name="num_waiters_in_place")
    )
    single_waiter_person_place = person_place_waiter[
        person_place_waiter["num_waiters_in_place"] == 1
    ][["person_id", "place_id"]]
    person_place_waiter_map = df[["person_id", "place_id", "waiter_id"]].drop_duplicates()
    single_waiter_records = single_waiter_person_place.merge(
        person_place_waiter_map, on=["person_id", "place_id"], how="left"
    )

    clients_only_this_waiter = (
        single_waiter_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_clients_only_this_waiter")
    )
    waiter_data = waiter_data.merge(
        clients_only_this_waiter, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_clients_only_this_waiter"] = (
        waiter_data["num_clients_only_this_waiter"].fillna(0).astype(int)
    )

    single_waiter_active_records = single_waiter_records[
        single_waiter_records["person_id"].isin(active_person_ids)
    ]
    active_clients_only_this_waiter = (
        single_waiter_active_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_active_clients_only_this_waiter")
    )
    waiter_data = waiter_data.merge(
        active_clients_only_this_waiter, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_active_clients_only_this_waiter"] = (
        waiter_data["num_active_clients_only_this_waiter"].fillna(0).astype(int)
    )

    person_waiter = (
        df.groupby("person_id")["waiter_id"].nunique().reset_index(name="num_waiters_total")
    )
    single_waiter_total_persons = person_waiter[person_waiter["num_waiters_total"] == 1][["person_id"]]
    person_waiter_total_map = df[["person_id", "waiter_id"]].drop_duplicates()
    single_waiter_total_records = single_waiter_total_persons.merge(
        person_waiter_total_map, on="person_id", how="left"
    )

    clients_single_waiter_total = (
        single_waiter_total_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_clients_single_waiter_total")
    )
    waiter_data = waiter_data.merge(
        clients_single_waiter_total, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_clients_single_waiter_total"] = (
        waiter_data["num_clients_single_waiter_total"].fillna(0).astype(int)
    )

    single_waiter_total_active_records = single_waiter_total_records[
        single_waiter_total_records["person_id"].isin(active_person_ids)
    ]
    active_clients_single_waiter_total = (
        single_waiter_total_active_records.groupby("waiter_id")["person_id"]
        .nunique()
        .rename("num_active_clients_single_waiter_total")
    )
    waiter_data = waiter_data.merge(
        active_clients_single_waiter_total, left_on="waiter_id", right_index=True, how="left"
    )
    waiter_data["num_active_clients_single_waiter_total"] = (
        waiter_data["num_active_clients_single_waiter_total"].fillna(0).astype(int)
    )

    waiter_data["share_clients_only_this_waiter"] = (
        waiter_data["num_clients_only_this_waiter"] / waiter_data["num_of_clients"]
    ).fillna(0)
    waiter_data["share_active_clients_only_this_waiter"] = (
        waiter_data["num_active_clients_only_this_waiter"]
        / waiter_data["num_of_active_clients"].replace(0, np.nan)
    ).fillna(0)
    waiter_data["share_clients_single_waiter_total"] = (
        waiter_data["num_clients_single_waiter_total"] / waiter_data["num_of_clients"]
    ).fillna(0)
    waiter_data["share_active_clients_single_waiter_total"] = (
        waiter_data["num_active_clients_single_waiter_total"]
        / waiter_data["num_of_active_clients"].replace(0, np.nan)
    ).fillna(0)

    # Transactions / clients where this waiter is the client's top_waiter_id
    df_top = df[["person_id", "waiter_id", "trn_id"]].merge(
        client_data[["top_waiter_id", "num_of_trn"]],
        left_on="person_id",
        right_index=True,
        how="left",
    )
    match_top = (df_top["waiter_id"] == df_top["top_waiter_id"]) & df_top[
        "top_waiter_id"
    ].notna()
    df_m = df_top.loc[match_top]
    trn_as_top = df_m.groupby("waiter_id")["trn_id"].nunique().rename(
        "num_of_trn_as_top_waiter"
    )
    clients_as_top = df_m.groupby("waiter_id")["person_id"].nunique().rename(
        "num_of_clients_as_top_waiter"
    )
    waiter_data = waiter_data.merge(trn_as_top, on="waiter_id", how="left")
    waiter_data = waiter_data.merge(clients_as_top, on="waiter_id", how="left")
    waiter_data["num_of_trn_as_top_waiter"] = (
        waiter_data["num_of_trn_as_top_waiter"].fillna(0).astype(int)
    )
    waiter_data["num_of_clients_as_top_waiter"] = (
        waiter_data["num_of_clients_as_top_waiter"].fillna(0).astype(int)
    )
    waiter_data["share_trn_as_top_waiter"] = (
        waiter_data["num_of_trn_as_top_waiter"]
        / waiter_data["num_of_trn"].replace(0, np.nan)
    ).fillna(0)
    trn_as_top_5 = (
        df_m.loc[df_m["num_of_trn"] > 4].groupby("waiter_id")["trn_id"].nunique()
    )
    trn_as_top_10 = (
        df_m.loc[df_m["num_of_trn"] > 9].groupby("waiter_id")["trn_id"].nunique()
    )
    denom_trn = waiter_data["num_of_trn"].replace(0, np.nan)
    waiter_data["share_trn_as_top_waiter_5"] = (
        waiter_data["waiter_id"].map(trn_as_top_5).fillna(0) / denom_trn
    ).fillna(0)
    waiter_data["share_trn_as_top_waiter_10"] = (
        waiter_data["waiter_id"].map(trn_as_top_10).fillna(0) / denom_trn
    ).fillna(0)
    waiter_data["share_clients_as_top_waiter"] = (
        waiter_data["num_of_clients_as_top_waiter"]
        / waiter_data["num_of_clients"].replace(0, np.nan)
    ).fillna(0)

    # Pooled share of distinct trn at places where this waiter works (vs each place's total trn)
    place_trn_total = df.groupby("place_id")["trn_id"].nunique().rename("place_trn_total")
    wp_trn = (
        df.groupby(["waiter_id", "place_id"])["trn_id"]
        .nunique()
        .reset_index(name="waiter_place_trn")
    )
    wp_trn = wp_trn.merge(
        place_trn_total, left_on="place_id", right_index=True, how="left"
    )
    share_place = wp_trn.groupby("waiter_id").agg(
        sum_waiter_trn=("waiter_place_trn", "sum"),
        sum_place_trn=("place_trn_total", "sum"),
    )
    share_place["share_trn_of_places_worked"] = share_place["sum_waiter_trn"] / share_place[
        "sum_place_trn"
    ].replace(0, np.nan)
    share_place = share_place[["share_trn_of_places_worked"]].reset_index()
    waiter_data = waiter_data.merge(share_place, on="waiter_id", how="left")

    # Transaction-weighted mean anomaly score over clients whose top_waiter is this waiter
    top_w = client_data.loc[
        client_data["top_waiter_id"].notna(),
        [
            "top_waiter_id",
            "num_of_trn",
            "share_top_waiter",
            "anomaly_score_iso",
            "anomaly_score_ocsvm",
            "anomaly_score_lof",
        ],
    ].copy()
    top_w["w_iso"] = top_w["num_of_trn"] * top_w["anomaly_score_iso"]
    top_w["w_ocsvm"] = top_w["num_of_trn"] * top_w["anomaly_score_ocsvm"]
    top_w["w_lof"] = top_w["num_of_trn"] * top_w["anomaly_score_lof"]
    weighted_top = top_w.groupby("top_waiter_id", as_index=False).agg(
        sum_trn=("num_of_trn", "sum"),
        sum_w_iso=("w_iso", "sum"),
        sum_w_ocsvm=("w_ocsvm", "sum"),
        sum_w_lof=("w_lof", "sum"),
        median_share_top_waiter_as_top_waiter=("share_top_waiter", "median"),
    )
    med_share_5 = (
        top_w.loc[top_w["num_of_trn"] > 4]
        .groupby("top_waiter_id", as_index=False)["share_top_waiter"]
        .median()
        .rename(columns={"share_top_waiter": "median_share_top_waiter_as_top_waiter_5"})
    )
    med_share_10 = (
        top_w.loc[top_w["num_of_trn"] > 9]
        .groupby("top_waiter_id", as_index=False)["share_top_waiter"]
        .median()
        .rename(columns={"share_top_waiter": "median_share_top_waiter_as_top_waiter_10"})
    )
    weighted_top = weighted_top.merge(med_share_5, on="top_waiter_id", how="left")
    weighted_top = weighted_top.merge(med_share_10, on="top_waiter_id", how="left")
    denom = weighted_top["sum_trn"].replace(0, np.nan)
    weighted_top["avg_weighted_anomaly_score_iso_as_top_waiter"] = (
        weighted_top["sum_w_iso"] / denom
    )
    weighted_top["avg_weighted_anomaly_score_ocsvm_as_top_waiter"] = (
        weighted_top["sum_w_ocsvm"] / denom
    )
    weighted_top["avg_weighted_anomaly_score_lof_as_top_waiter"] = (
        weighted_top["sum_w_lof"] / denom
    )
    weighted_top = weighted_top.rename(columns={"top_waiter_id": "waiter_id"})
    waiter_data = waiter_data.merge(
        weighted_top[
            [
                "waiter_id",
                "avg_weighted_anomaly_score_iso_as_top_waiter",
                "avg_weighted_anomaly_score_ocsvm_as_top_waiter",
                "avg_weighted_anomaly_score_lof_as_top_waiter",
                "median_share_top_waiter_as_top_waiter",
                "median_share_top_waiter_as_top_waiter_5",
                "median_share_top_waiter_as_top_waiter_10",
            ]
        ],
        on="waiter_id",
        how="left",
    )

    return waiter_data

# Match `config.load_data` client filters (adjust to mirror your modeling pipeline).
ACTIVITY_STATE = 1
DAYS_VISITS = 1

parquet_dir = Path(DATA_PATH)
scores_path = ROOT / "client_anomaly_scores.csv"
out_path = parquet_dir / "waiter_level_features.parquet"

df = pd.read_parquet(parquet_dir / "processed_transactions.parquet", engine="pyarrow")

client_data = pd.read_parquet(parquet_dir / "client_level_features.parquet", engine="pyarrow")
client_data = client_data[
    (client_data["num_of_trn"] > ACTIVITY_STATE) & (client_data["days_visits"] > DAYS_VISITS)
].copy()
client_data["is_fraud"] = client_data["person_id"].isin(FRAUD_IDS).astype(int)
client_data = client_data.set_index("person_id")

score_df = pd.read_csv(scores_path)
score_df = score_df.rename(
    columns={
        "iso_score": "anomaly_score_iso",
        "ocsvm_score": "anomaly_score_ocsvm",
        "lof_score": "anomaly_score_lof",
    }
).set_index("person_id")

client_data = client_data.join(
    score_df[["anomaly_score_iso", "anomaly_score_ocsvm", "anomaly_score_lof"]],
    how="inner",
)
client_data = client_data.dropna(
    subset=["anomaly_score_iso", "anomaly_score_ocsvm", "anomaly_score_lof"]
)

df = df[df["person_id"].isin(client_data.index)].copy()

waiter_level = build_waiter_data(df, client_data)

ww_scores_path = ROOT / "waiter_week_anomaly_scores.csv"
ww_scores = pd.read_csv(ww_scores_path)
share_weeks = _share_anomaly_weeks_from_waiter_week_scores(ww_scores)
waiter_level = waiter_level.merge(share_weeks, on="waiter_id", how="left")


waiter_level.to_parquet(out_path, index=False, engine="pyarrow")
print(f"Wrote {len(waiter_level)} rows to {out_path}")
waiter_level.head()

Num of fraud waiter ids:  30
Wrote 5353 rows to /Users/yuliia/Documents/Fraud-Detection/parquet/waiter_level_features.parquet


,waiter_id,iso_90,ocsvm_90,lof_90,num_of_trn,num_of_clients,working_days,max_client_anomaly_score_iso,median_client_anomaly_score_iso,avg_client_anomaly_score_iso,...,share_anomaly_weeks_lof,avg_anomaly_score_iso,avg_anomaly_score_ocsvm,avg_anomaly_score_lof,max_anomaly_week_score_iso,max_anomaly_week_score_ocsvm,max_anomaly_week_score_lof,median_anomaly_week_score_iso,median_anomaly_week_score_ocsvm,median_anomaly_week_score_lof
0,539f5da6c76b7cd7fa3d859c_0,0.487513,-0.009403,1.145461,5,4,1,0.513029,0.447418,0.461155,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,539f5da6c76b7cd7fa3d859c_0.31,0.452902,-0.015376,1.011608,2,2,2,0.453332,0.451181,0.451181,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,539f5da6c76b7cd7fa3d859c_00,0.487912,-0.016463,1.120324,136,131,25,0.517228,0.455381,0.458995,...,0.000000,0.385771,-0.002258,1.097931,0.413903,0.004916,1.144959,0.382671,-0.001464,1.100808
3,539f5da6c76b7cd7fa3d859c_001,0.484906,-0.014343,1.104510,2021,1782,290,0.571213,0.449520,0.454155,...,0.140845,0.391090,-0.011047,1.109257,0.616688,0.002139,1.382105,0.376929,-0.011059,1.077615
4,539f5da6c76b7cd7fa3d859c_00131,0.465253,-0.041191,0.998481,2,2,2,0.468251,0.453261,0.453261,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
waiter_level

,waiter_id,iso_90,ocsvm_90,lof_90,num_of_trn,num_of_clients,working_days,is_fraud,num_of_active_clients,num_clients_only_this_waiter,num_active_clients_only_this_waiter,num_clients_single_waiter_total,num_active_clients_single_waiter_total,share_clients_only_this_waiter,share_active_clients_only_this_waiter,share_clients_single_waiter_total,share_active_clients_single_waiter_total,share_anomaly_weeks_iso,share_anomaly_weeks_ocsvm,share_anomaly_weeks_lof
0,539f5da6c76b7cd7fa3d859c_0,0.542703,-0.018725,1.160229,5,4,1,False,2,4,2,0,0,1.000000,1.000000,0.0,0.0,NaN,NaN,NaN
1,539f5da6c76b7cd7fa3d859c_0.31,0.526659,-0.031042,1.018016,2,2,2,False,2,0,0,0,0,0.000000,0.000000,0.0,0.0,NaN,NaN,NaN
2,539f5da6c76b7cd7fa3d859c_00,0.525061,-0.033092,1.140944,136,131,25,False,72,81,46,0,0,0.618321,0.638889,0.0,0.0,0.000000,0.142857,0.142857
3,539f5da6c76b7cd7fa3d859c_001,0.529399,-0.028704,1.111019,2021,1782,290,False,961,1110,501,0,0,0.622896,0.521332,0.0,0.0,0.027778,0.041667,0.083333
4,539f5da6c76b7cd7fa3d859c_00131,0.491820,-0.082152,1.096395,2,2,2,False,0,1,0,0,0,0.500000,0.000000,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5348,675ac8c769a6f18612504814_17494,0.536910,-0.041957,1.160229,90,72,7,False,50,65,43,0,0,0.902778,0.860000,0.0,0.0,0.000000,0.000000,0.000000
5349,675ac8c769a6f18612504814_18352,0.516226,-0.047856,1.140218,79,74,5,False,54,68,48,0,0,0.918919,0.888889,0.0,0.0,0.000000,0.000000,0.000000
5350,675ac8c769a6f18612504814_19362,0.543888,-0.010709,1.101054,22,19,2,False,14,17,12,0,0,0.894737,0.857143,0.0,0.0,0.000000,0.000000,0.000000
5351,6762deb569a624876c17774a_acherepania,0.533562,-0.062160,1.167703,1,1,1,False,0,1,0,0,0,1.000000,0.000000,0.0,0.0,NaN,NaN,NaN
